In [3]:
import h5py
import numpy as np
from illustris_python.groupcat import loadSingle, loadHeader
import illustris_python as il
import matplotlib.pyplot as plt
import matplotlib 
import mpl_toolkits.mplot3d as mpl3
from matplotlib.gridspec import GridSpec
import requests
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import tempfile
import contextlib
import os


In [4]:
# Constante de Hubble en unidades de [km/s / Mpc]:
H0 = 67.74
h = H0 / 100 

# Tamaño del box de simulación
box_size = 204.98124  # Mpc/h

# Path de las simulaciones TNG300-1 en este directorio:
myBasePath = '../sims.TNG/TNG300-1/output' 

# PASO 1: BAJAR HALOS QUE A Z=0 TENGAN $M_{200}$ EN [Msun/h] MAYOR A 13.5

In [5]:
# Traemos algunas propiedades de TODOS los halos de las TNG300
data_halos = il.groupcat.loadHalos(myBasePath, 99, fields=['GroupFirstSub', 'Group_M_Crit200', 'Group_R_Crit200', 'GroupPos', 'GroupVel'])

In [6]:
# Armamos un DataFrame con la info bajada en las unidades correctas
halos = pd.DataFrame({
    'id_z0': np.arange(len(data_halos['GroupPos'])),
    'id_central_galaxy_z0': data_halos['GroupFirstSub'],
    'm200_z0': np.log10(data_halos['Group_M_Crit200'] * 1e10), # Msun/h
    'r200_z0': data_halos['Group_R_Crit200'] / 1000, # cMpc/h
    'pos_x_z0': data_halos['GroupPos'][:, 0] / 1000, # cMpc/h
    'pos_y_z0': data_halos['GroupPos'][:, 1] / 1000, # cMpc/h
    'pos_z_z0': data_halos['GroupPos'][:, 2] / 1000, # cMpc/h
    'vel_x_z0': data_halos['GroupVel'][:, 0], # km/s
    'vel_y_z0': data_halos['GroupVel'][:, 1], # km/s
    'vel_z_z0': data_halos['GroupVel'][:, 2] # km/s
})

# Seleccionamos el treshold en M_200
halos = halos[halos['m200_z0'] >= 13.5]
len(halos)

754

# PASO 2: CRITERIO DE AISLAMIENTO

In [7]:
def periodic_distance(pos1, pos2, box_size):
    """
    Calcula la distancia periódica entre dos posiciones.
    
    Parameters:
    -----------
    pos1 : array (3,)
        Posición del primer objeto [x, y, z]
    pos2 : array (N, 3) 
        Posiciones de N objetos
    box_size : float
        Tamaño de la caja de simulación
        
    Returns:
    --------
    distances : array (N,)
        Distancias periódicas
    """
    # Diferencia en cada dimensión
    delta = np.abs(pos2 - pos1)
    
    # Aplicar condición de periodicidad: si delta > box_size/2, usar camino corto
    delta = np.minimum(delta, box_size - delta)
    
    # Distancia euclidiana 3D
    distances = np.sqrt(np.sum(delta**2, axis=1))
    
    return distances

In [8]:
def filter_isolated_halos(halos_df, data_halos, box_size, verbose=True):
    '''
    Filtra halos que cumplen el criterio de aislamiento simple:
    Descarta un halo si tiene otro halo con masa > 10% de su m200 dentro de 5*r200
    
    INCLUYE CORRECCIÓN DE PERIODICIDAD para condiciones de frontera.
    
    Parameters:
    -----------
    halos_df : DataFrame
        DataFrame con los halos seleccionados
    data_halos : dict
        Diccionario con todos los halos de la simulación
    box_size : float
        Tamaño de la caja de simulación (Mpc/h)
    verbose : bool
        Si imprimir información del progreso
        
    Returns:
    --------
    DataFrame con halos aislados
    '''
    
    # Convertir masas de data_halos a log10(M_sun) como en tu DataFrame
    all_masses_log = np.log10(data_halos['Group_M_Crit200'] * 1e10) # Msun/h
    all_positions_mpc = data_halos['GroupPos'] / 1000  # Mpc/h
    
    # Filtrar halos válidos (masa > 0)
    valid_mask = data_halos['Group_M_Crit200'] > 0
    all_masses_log = all_masses_log[valid_mask]
    all_positions_mpc = all_positions_mpc[valid_mask]
    
    if verbose:
        print(f'Total de halos válidos en data_halos: {len(all_masses_log)}')
        print(f'Halos a verificar: {len(halos_df)}')
    
    isolated_indices = []
    
    for idx, halo in halos_df.iterrows():
        halo_id = halo['id_z0'] # ID del halo actual
        halo_mass = halo['m200_z0'] # log10(Msun/h)
        halo_radius = halo['r200_z0'] # Mpc/h
        halo_pos = np.array([halo['pos_x_z0'], halo['pos_y_z0'], halo['pos_z_z0']]) # Mpc/h
        
        # Masa mínima para considerar (10% de la masa del halo)
        min_mass_log = halo_mass + np.log10(0.1) # log10(0.1 * M_halo)
        
        # Radio de búsqueda      
        search_radius = 5 * halo_radius
        
        # Encontrar halos más masivos que el 10% EXCLUYENDO el halo actual
        massive_mask = (all_masses_log > min_mass_log) & (np.arange(len(all_masses_log)) != halo_id)
        
        if np.any(massive_mask):
            # Posiciones de halos más masivos
            massive_positions = all_positions_mpc[massive_mask]
            
            # Calcular distancias PERIÓDICAS
            distances = periodic_distance(halo_pos, massive_positions, box_size)
            
            # Verificar si algún halo masivo está dentro del radio de búsqueda
            close_massive_halos = np.any(distances < search_radius)
            
            if not close_massive_halos:
                # No hay halos masivos cerca, está aislado
                isolated_indices.append(idx)
        else:
            # No hay halos más masivos, está aislado
            isolated_indices.append(idx)
    
    isolated_halos = halos_df.loc[isolated_indices].copy()
    
    if verbose:
        print(f'Halos aislados: {len(isolated_halos)}')
        print(f'Halos descartados: {len(halos_df) - len(isolated_halos)}')
        print(f'Fracción aislada: {len(isolated_halos)/len(halos_df):.1%}')
    
    return isolated_halos

In [9]:
isolated_halos = filter_isolated_halos(halos, data_halos, box_size)

Total de halos válidos en data_halos: 11461519
Halos a verificar: 754
Halos aislados: 549
Halos descartados: 205
Fracción aislada: 72.8%


# PASO 3: TRAEMOS TODAS LAS GALAXIAS CON $M_{\star}$ EN [Msun/h] MAYOR A 9.

In [10]:
data_subhalos = il.groupcat.loadSubhalos(myBasePath, 99, fields=['SubhaloFlag', 'SubhaloMassType', 'SubhaloPos', 'SubhaloVel'])

In [11]:
subhalos = pd.DataFrame({
    'sub_id_z0': np.arange(len(data_subhalos['SubhaloPos'])),
    'sub_mstar_z0': np.log10(data_subhalos['SubhaloMassType'][:, 4] * 1e10), # Msun/h
    'sub_pos_x_z0': data_subhalos['SubhaloPos'][:, 0] / 1000, # cMpc/h
    'sub_pos_y_z0': data_subhalos['SubhaloPos'][:, 1] / 1000, # cMpc/h
    'sub_pos_z_z0': data_subhalos['SubhaloPos'][:, 2] / 1000, # cMpc/h
    'sub_vel_x_z0': data_subhalos['SubhaloVel'][:, 0], # km/s
    'sub_vel_y_z0': data_subhalos['SubhaloVel'][:, 1], # km/s
    'sub_vel_z_z0': data_subhalos['SubhaloVel'][:, 2], # km/s
    'flag': data_subhalos['SubhaloFlag']
})

# Seleccionamos el treshold en M_star
subhalos = subhalos[subhalos['sub_mstar_z0'] >= 9.]

# Excluimos los halos con flag == 0
subhalos = subhalos[subhalos['flag'] != 0.]
len(subhalos)

237222

In [12]:
subhalos

,sub_id_z0,sub_mstar_z0,sub_pos_x_z0,sub_pos_y_z0,sub_pos_z_z0,sub_vel_x_z0,sub_vel_y_z0,sub_vel_z_z0,flag
0,0,12.725053,43.718811,48.813641,147.594955,472.196198,450.850006,-260.746918,True
1,1,12.404258,45.442272,51.850201,146.416504,-209.056656,-735.888916,400.641724,True
2,2,11.758653,44.490761,49.091713,147.870575,2021.729492,1495.440186,-1797.082153,True
3,3,11.378576,43.820786,50.939400,147.711044,925.150391,-473.445465,-275.925934,True
4,4,11.343598,44.302578,49.630974,147.869492,-260.214630,-2221.625244,-563.641296,True
...,...,...,...,...,...,...,...,...,...
4034415,4034415,9.076360,149.481995,131.930542,59.186455,-64.029015,-127.274849,-165.122711,True
4042777,4042777,9.095128,193.151932,142.552689,181.271729,208.924545,-114.543533,144.491577,True
4169607,4169607,9.080675,64.253746,45.295513,28.788446,303.923401,204.920273,68.553764,True
4256899,4256899,9.122880,184.153030,31.699749,187.823334,-194.852570,38.616081,-290.989899,True


In [13]:
# Calculamos, para cada halo la distancia 3D a todas las galaxias. Si esta es menor a 5*r200 entonces 
# sí la agregamos al DataFrame 'data'

data = pd.DataFrame()

# Iteramos sobre cada halo
for _, halo in tqdm(isolated_halos.iterrows(), total=len(isolated_halos)):
    
    # Definimos propiedades del halo actual
    halo_pos_x_z0 = halo['pos_x_z0']  # cMpc/h
    halo_pos_y_z0 = halo['pos_y_z0']  # cMpc/h
    halo_pos_z_z0 = halo['pos_z_z0']  # cMpc/h
    halo_vel_x_z0 = halo['vel_x_z0']  # km/s
    halo_vel_y_z0 = halo['vel_y_z0']  # km/s
    halo_vel_z_z0 = halo['vel_z_z0']  # km/s
    halo_r200_z0 = halo['r200_z0']   # cMpc/h
    halo_m200_z0 = halo['m200_z0']   # Msun/h
    halo_id_z0 = halo['id_z0']
    halo_id_central_galaxy_z0 = halo['id_central_galaxy_z0']
    
    # Calculamos la distancia de cada subhalo al halo actual SIN PERIODICIDAD
    dx = np.abs(subhalos['sub_pos_x_z0'] - halo_pos_x_z0) # cMpc/h
    dy = np.abs(subhalos['sub_pos_y_z0'] - halo_pos_y_z0) # cMpc/h
    dz = np.abs(subhalos['sub_pos_z_z0'] - halo_pos_z_z0) # cMpc/h
    
    # APLICAMOS PERIODICIDAD: si la distancia > box_size/2, entonces usamos el camino más corto
    dx = np.minimum(dx, box_size - dx) # cMpc/h
    dy = np.minimum(dy, box_size - dy) # cMpc/h
    dz = np.minimum(dz, box_size - dz) # cMpc/h
    
    # Calculamos distancia 3D periódica de cada subhalo al halo actual
    subhalos['sub_dist_3d_z0'] = np.sqrt(dx**2 + dy**2 + dz**2)  # cMpc/h
    
    # Filtramos los subhalos que están dentro de la esfera de 5 * r200
    subhalos_in_range = subhalos[subhalos['sub_dist_3d_z0'] <= 5 * halo_r200_z0].copy()
    
    if len(subhalos_in_range) > 0:
        
        # Agregamos la información del halo a cada subhalo en rango
        subhalos_in_range['halo_id_z0'] = halo_id_z0
        subhalos_in_range['halo_id_central_galaxy_z0'] = halo_id_central_galaxy_z0
        subhalos_in_range['halo_r200_z0'] = halo_r200_z0
        subhalos_in_range['halo_m200_z0'] = halo_m200_z0
        subhalos_in_range['halo_pos_x_z0'] = halo_pos_x_z0
        subhalos_in_range['halo_pos_y_z0'] = halo_pos_y_z0
        subhalos_in_range['halo_pos_z_z0'] = halo_pos_z_z0
        subhalos_in_range['halo_vel_x_z0'] = halo_vel_x_z0
        subhalos_in_range['halo_vel_y_z0'] = halo_vel_y_z0
        subhalos_in_range['halo_vel_z_z0'] = halo_vel_z_z0
        
        # Reordenamos las columnas para que la info del halo vaya primero
        halo_columns = ['halo_id_z0']
        other_columns = [col for col in subhalos_in_range.columns if col not in halo_columns]
        subhalos_in_range = subhalos_in_range[halo_columns + other_columns]
        
        # Añadimos los subhalos en rango al DataFrame 'data'
        data = pd.concat([data, subhalos_in_range], ignore_index=True)

100%|██████████| 549/549 [00:09<00:00, 58.18it/s]


# PASO 4: CORREGIMOS LAS POSICIONES DE LAS GALAXIAS POR PERIODICIDAD

In [14]:
# Inicializamos la columna 'pos_corr' con NaN
data['pos_corr'] = np.nan

# Para cada subhalo en data, calculamos las posiciones corregidas
for idx, row in data.iterrows():
    
    # Posiciones originales del subhalo
    sub_pos_x = row['sub_pos_x_z0']
    sub_pos_y = row['sub_pos_y_z0'] 
    sub_pos_z = row['sub_pos_z_z0']
    
    # Posiciones del halo al que pertenece
    halo_pos_x = row['halo_pos_x_z0']
    halo_pos_y = row['halo_pos_y_z0']
    halo_pos_z = row['halo_pos_z_z0']
    
    # Calculamos las diferencias sin periodicidad
    dx_raw = sub_pos_x - halo_pos_x
    dy_raw = sub_pos_y - halo_pos_y
    dz_raw = sub_pos_z - halo_pos_z
    
    # Aplicamos correcciones periódicas
    pos_x_corrected = sub_pos_x
    pos_y_corrected = sub_pos_y
    pos_z_corrected = sub_pos_z
    
    # Lista para registrar en qué ejes se hizo corrección
    corrections = []
    
    # Si la distancia en X entre el halo y el subhalo es mayor a box_size/2 se corrige la posición en X.
    # Cómo hicimos subhalo - halo: positivo es el subhalo por delante, negativo es el subhalo por detrás.
    
    # dx_raw > +box_size/2: subhalo muy a la derecha → lo traemos por la izquierda (restamos box_size)
    # dx_raw < -box_size/2: subhalo muy a la izquierda → lo traemos por la derecha (sumamos box_size)
    
    # Corrección en X
    if dx_raw > box_size/2:
        pos_x_corrected -= box_size
        corrections.append('X')
    elif dx_raw < -box_size/2:
        pos_x_corrected += box_size
        corrections.append('X')
        
    # Corrección en Y
    if dy_raw > box_size/2:
        pos_y_corrected -= box_size
        corrections.append('Y')
    elif dy_raw < -box_size/2:
        pos_y_corrected += box_size
        corrections.append('Y')
        
    # Corrección en Z  
    if dz_raw > box_size/2:
        pos_z_corrected -= box_size
        corrections.append('Z')
    elif dz_raw < -box_size/2:
        pos_z_corrected += box_size
        corrections.append('Z')
    
    # Actualizamos las posiciones en el DataFrame data
    data.loc[idx, 'sub_pos_x_z0'] = pos_x_corrected
    data.loc[idx, 'sub_pos_y_z0'] = pos_y_corrected
    data.loc[idx, 'sub_pos_z_z0'] = pos_z_corrected
    
    # Si hubo correcciones, registramos en qué ejes
    if corrections:
        data.loc[idx, 'pos_corr'] = ''.join(corrections)

print(f"Subhalos con correcciones: {data['pos_corr'].notna().sum()}")
print(f"\nDistribución de correcciones:")
print(data['pos_corr'].value_counts().sort_index())

Subhalos con correcciones: 152

Distribución de correcciones:
pos_corr
X    21
Y    51
Z    80
Name: count, dtype: int64


In [15]:
data = data.drop(['flag', 'pos_corr'], axis=1)

# PASO 4: MERGER TREE DE LOS SUBHALOS

In [16]:
def apply_periodic_correction(sub_pos, halo_pos, box_size):
    """
    Aplica corrección de periodicidad a las posiciones del subhalo
    respecto al halo en cada snapshot.
    
    Parameters:
    -----------
    sub_pos : array (x, y, z)
        Posición del subhalo
    halo_pos : array (x, y, z)
        Posición del halo
    box_size : float
        Tamaño de la caja de simulación
        
    Returns:
    --------
    corrected_pos : array (x, y, z)
        Posición corregida del subhalo
    """
    corrected_pos = sub_pos.copy()
    
    # Calcular diferencias
    dx = sub_pos[0] - halo_pos[0]
    dy = sub_pos[1] - halo_pos[1]
    dz = sub_pos[2] - halo_pos[2]
    
    # Corrección en X
    if dx > box_size/2:
        corrected_pos[0] -= box_size
    elif dx < -box_size/2:
        corrected_pos[0] += box_size
        
    # Corrección en Y
    if dy > box_size/2:
        corrected_pos[1] -= box_size
    elif dy < -box_size/2:
        corrected_pos[1] += box_size
        
    # Corrección en Z
    if dz > box_size/2:
        corrected_pos[2] -= box_size
    elif dz < -box_size/2:
        corrected_pos[2] += box_size
        
    return corrected_pos

In [17]:
def load_halo_evolution(myBasePath, halo_id_z0, id_central_galaxy_z0):
    """
    Carga la evolución del halo desde z=0 hasta el snapshot más antiguo disponible.
    
    Returns:
    --------
    dict con 'pos_x', 'pos_y', 'pos_z', 'r200', 'm200', 'snapshots'
    """
    # Cargar posiciones del halo (usando el subhalo central)
    try:
        central_galaxy_info = il.sublink.loadTree(
            myBasePath, 99, id_central_galaxy_z0,
            fields=['SubhaloGrNr', 'SubhaloPos', 'SnapNum'], 
            onlyMPB=True
        )
        
        halo_pos_x = central_galaxy_info['SubhaloPos'][:, 0] / 1000  # cMpc/h
        halo_pos_y = central_galaxy_info['SubhaloPos'][:, 1] / 1000  # cMpc/h
        halo_pos_z = central_galaxy_info['SubhaloPos'][:, 2] / 1000  # cMpc/h
        snapshots = central_galaxy_info.get('SnapNum', np.arange(99, 99-len(halo_pos_x), -1))
        
    except Exception as e:
        return None
    
    # Cargar r200 y m200 para cada snapshot
    halo_r200 = []
    halo_m200 = []
    halo_id = []
    for snap in snapshots:
        try:            
            aux = central_galaxy_info['SubhaloGrNr'][np.where(central_galaxy_info['SnapNum'] == snap)[0][0]]
            halo_id.append(aux)
            halo_info = il.groupcat.loadSingle(myBasePath, snap, haloID=aux)
            halo_r200.append(round(halo_info['Group_R_Crit200'] / 1000, 5))  # cMpc/h
            halo_m200.append(round(np.log10(halo_info['Group_M_Crit200'] * 1e10), 5))  # Msun/h
        except Exception:
            halo_r200.append(np.nan)
            halo_m200.append(np.nan)
            halo_id.append(np.nan)
    
    return {
        'pos_x': np.array(halo_pos_x),
        'pos_y': np.array(halo_pos_y),
        'pos_z': np.array(halo_pos_z),
        'r200': np.array(halo_r200),
        'm200': np.array(halo_m200),
        'snapshots': np.array(snapshots),
        'haloID': halo_id
    }

In [19]:
K_B = 1.38 * 1e-16 # erg/K
def load_subhalo_evolution(myBasePath, sub_id_z0, halo_evolution, plotear = False):
    """
    Carga la evolución del subhalo y aplica correcciones de periodicidad.
    
    Returns:
    --------
    dict con 'pos_x', 'pos_y', 'pos_z' corregidos, o None si falla
    """
   # try:
    # Cargar el MergerTree del subhalo
    tree = il.sublink.loadTree(
        myBasePath, 99, sub_id_z0, 
        fields=['SubhaloGrNr', 'SubhaloPos', 'SnapNum','SubhaloHalfmassRad','SubhaloIDRaw'], 
        onlyMPB=True
    )

    # Si el árbol está vacío, retornar None
    if tree is None or len(tree['SubhaloPos']) == 0:
        return None

    # Posiciones originales del subhalo
    sub_pos_x = tree['SubhaloPos'][:, 0] / 1000  # cMpc/h
    sub_pos_y = tree['SubhaloPos'][:, 1] / 1000  # cMpc/h
    sub_pos_z = tree['SubhaloPos'][:, 2] / 1000  # cMpc/h
    sub_halfMassRad = tree['SubhaloHalfmassRad'] / 1000  # cMpc/h
    sub_snapshots = tree.get('SnapNum', np.arange(99, 99-len(sub_pos_x), -1))

    # Arrays para posiciones corregidas
    pos_x_corrected = []
    pos_y_corrected = []
    pos_z_corrected = []

    #%gas_num_density = np.ones(len(sub_snapshots)) * (-99)
    #%gas_mass_density = np.ones(len(sub_snapshots)) * (-99) 
    subhalo_gas_num = np.ones((len(sub_snapshots), 3)) * (-99) 
    halo_gas_num = np.ones((len(sub_snapshots), 3)) * (-99)
    subhalo_gas_mass = np.ones((len(sub_snapshots), 3)) * (-99) 
    halo_gas_mass = np.ones((len(sub_snapshots), 3)) * (-99) 
    subhalo_gas_intEnergy = np.ones((len(sub_snapshots), 3)) * (-99) 
    halo_gas_intEnergy = np.ones((len(sub_snapshots), 3)) * (-99) 
    subhalo_gas_elecAbundance = np.ones((len(sub_snapshots), 3)) * (-99) 
    halo_gas_elecAbundance = np.ones((len(sub_snapshots), 3)) * (-99)
    subhalo_gas_T = np.ones((len(sub_snapshots), 3)) * (-99) 
    halo_gas_T = np.ones((len(sub_snapshots), 3)) * (-99)
    
    # Aplicar corrección de periodicidad y estimar la densidad de gas en cada snapshot
    for i, snap in enumerate(sub_snapshots[:]):
        if (i%10) == 0: print('Snapshot ' + str(snap) + '. Subhalo ' + str(sub_id_z0))
        # Encontrar el índice correspondiente en halo_evolution
        halo_idx = np.where(halo_evolution['snapshots'] == snap)[0]
        haloID = halo_evolution['haloID'][halo_idx[0]]
        
        if len(halo_idx) > 0:
            halo_idx = halo_idx[0]

            # Posición del halo en este snapshot
            halo_pos = np.array([
                halo_evolution['pos_x'][halo_idx],
                halo_evolution['pos_y'][halo_idx],
                halo_evolution['pos_z'][halo_idx]
            ])

            # Posición del subhalo en este snapshot
            sub_pos = np.array([sub_pos_x[i], sub_pos_y[i], sub_pos_z[i]])

            # Aplicar corrección de periodicidad
            corrected_pos = apply_periodic_correction(sub_pos, halo_pos, box_size)

            pos_x_corrected.append(corrected_pos[0])
            pos_y_corrected.append(corrected_pos[1])
            pos_z_corrected.append(corrected_pos[2])
        else:
            pos_x_corrected.append(sub_pos_x[i])
            pos_y_corrected.append(sub_pos_y[i])
            pos_z_corrected.append(sub_pos_z[i])
            
        
        flag = False                
        niter = 0
        while (flag == False) & (niter < 5):
            try:
                # Me guardo todas las particulas de gas del subhalo (el subhaloIDRaw es el subfindID + snapnum*1e12)
                #%gas_subhalo = get(get_url('subhalos', int(tree['SubhaloIDRaw'][i] - snap*1e12), snap) + '/' + 'cutout.hdf5', 
                #%   {'gas':'coordinates,density,ElectronAbundance,InternalEnergy,masses'})

                gas_subhalo = il.snapshot.loadSubhalo(myBasePath, snap, int(tree['SubhaloIDRaw'][i] - snap*1e12), 'gas',
                                                      fields = ['Coordinates','Density','ElectronAbundance','InternalEnergy','Masses'])

                # Me guardo todas las particulas de gas del halo
                #%gas_halo = get(get_url('halos', haloID, snap) + '/' + 'cutout.hdf5', {'gas':'coordinates,density,ElectronAbundance,InternalEnergy,masses'})
                gas_halo = il.snapshot.loadHalo(myBasePath, snap, haloID, 'gas',
                                                      fields = ['Coordinates','Density','ElectronAbundance','InternalEnergy','Masses'])

                flag = True
            except:
                gas_subhalo = []
                gas_halo = []
                niter += 1
                print('Fallo ' + str(niter) + ' en las TNG al bajar el snapshot ' + str(snap))
            
        if (gas_halo['count'] > 0): # Si no hay gas en el subhalo no hago nada
            # Me quedo con todo lo que este adentro de 2* galaxy-half-mass-radii
            aux_dist = (gas_halo['Coordinates'][:,0]/1e3 - sub_pos_x[i])**2 + (gas_halo['Coordinates'][:,1]/1e3 - sub_pos_y[i])**2 + (gas_halo['Coordinates'][:,2]/1e3 - sub_pos_z[i])**2
            aux_dist = np.sqrt(aux_dist) # MPc/h
            aux_ang = (gas_halo['Coordinates'][:,0]/1e3 - sub_pos_x[i]) * (halo_pos[0] - sub_pos_x[i]) + (gas_halo['Coordinates'][:,1]/1e3 - sub_pos_y[i]) * (halo_pos[1] - sub_pos_y[i]) + (gas_halo['Coordinates'][:,2]/1e3 - sub_pos_z[i]) * (halo_pos[2] - sub_pos_z[i])
            halo_ind_inside_2hmr = np.where(aux_dist < 2*sub_halfMassRad[i])[0]
            halo_ind_inside_fix = np.where(aux_dist < (50/1000))[0]
            halo_ind_bet_2_3 = np.where( (aux_dist > 2*sub_halfMassRad[i]) & (aux_dist < 3*sub_halfMassRad[i]) & (aux_ang > 0) )[0] # entre 2 y 3 hmf, y en la cara cercana al cumulo

            halo_gas_num[i,0] = len(halo_ind_inside_fix)
            halo_gas_mass[i,0] = np.sum(gas_halo['Masses'][halo_ind_inside_fix])
            halo_gas_intEnergy[i,0] = np.sum(gas_halo['InternalEnergy'][halo_ind_inside_fix])
            halo_gas_elecAbundance[i,0] = np.sum(gas_halo['ElectronAbundance'][halo_ind_inside_fix])
            mu = (4 * gas_halo['Masses'][halo_ind_inside_fix]) / (1 + 3*0.76 + 4*0.76*gas_halo['ElectronAbundance'][halo_ind_inside_fix]) # auxiliar mu estimation
            halo_gas_T[i,0] = np.sum((5/3 - 1) * gas_halo['InternalEnergy'][halo_ind_inside_fix] * mu / K_B)
            
            halo_gas_num[i,1] = len(halo_ind_inside_2hmr)
            halo_gas_mass[i,1] = np.sum(gas_halo['Masses'][halo_ind_inside_2hmr])
            halo_gas_intEnergy[i,1] = np.sum(gas_halo['InternalEnergy'][halo_ind_inside_2hmr])
            halo_gas_elecAbundance[i,1] = np.sum(gas_halo['ElectronAbundance'][halo_ind_inside_2hmr])
            mu = (4 * gas_halo['Masses'][halo_ind_inside_2hmr]) / (1 + 3*0.76 + 4*0.76*gas_halo['ElectronAbundance'][halo_ind_inside_2hmr]) # auxiliar mu estimation
            halo_gas_T[i,1] = np.sum((5/3 - 1) * gas_halo['InternalEnergy'][halo_ind_inside_2hmr] * mu / K_B)
            
            halo_gas_num[i,2] = len(halo_ind_bet_2_3)
            halo_gas_mass[i,2] = np.sum(gas_halo['Masses'][halo_ind_bet_2_3])
            halo_gas_intEnergy[i,2] = np.sum(gas_halo['InternalEnergy'][halo_ind_bet_2_3])
            halo_gas_elecAbundance[i,2] = np.sum(gas_halo['ElectronAbundance'][halo_ind_bet_2_3])
            mu = (4 * gas_halo['Masses'][halo_ind_bet_2_3]) / (1 + 3*0.76 + 4*0.76*gas_halo['ElectronAbundance'][halo_ind_bet_2_3]) # auxiliar mu estimation
            halo_gas_T[i,2] = np.sum((5/3 - 1) * gas_halo['InternalEnergy'][halo_ind_bet_2_3] * mu / K_B)
            
            #print('El halo SI tiene gas en el snapshot ' + str(snap) + ' ' + str(len(halo_ind_inside_fix)))
            
            if plotear:
                fig, ax = plt.subplots(1,1)
                rand_ind = np.random.randint(0, len(gas_halo[0]), 10000)
                ax.scatter(gas_halo[0][rand_ind,0]/1e3, gas_halo[0][rand_ind,1]/1e3, color = 'black', s = 5)
                ax.scatter(gas_halo[0][halo_ind_inside_fix,0]/1e3, gas_halo[0][halo_ind_inside_fix,1]/1e3, color = 'darkcyan', s = 2, marker = '*')
                r200_circ = plt.Circle((halo_pos[0], halo_pos[1]), halo_evolution['r200'][halo_idx], color='darkcyan', fill=False)
                ax.add_patch(r200_circ)
        else:
            #print('El halo no tiene gas en el snapshot ' + str(snap))
            halo_gas_num[i,0] = 0
            halo_gas_mass[i,0] = 0
            halo_gas_intEnergy[i,0] = 0
            halo_gas_elecAbundance[i,0] = 0
            halo_gas_T[i,0] = 0
            
            halo_gas_num[i,1] = 0
            halo_gas_mass[i,1] = 0
            halo_gas_intEnergy[i,1] = 0
            halo_gas_elecAbundance[i,1] = 0
            halo_gas_T[i,1] = 0
            
            halo_gas_num[i,2] = 0
            halo_gas_mass[i,2] = 0
            halo_gas_intEnergy[i,2] = 0
            halo_gas_elecAbundance[i,2] = 0
            halo_gas_T[i,2] = 0
        
        if (gas_subhalo['count'] > 0):
            # Me quedo con todo lo que este adentro de 2* galaxy-half-mass-radii
            aux_dist = (gas_subhalo['Coordinates'][:,0]/1e3 - sub_pos_x[i])**2 + (gas_subhalo['Coordinates'][:,1]/1e3 - sub_pos_y[i])**2 + (gas_subhalo['Coordinates'][:,2]/1e3 - sub_pos_z[i])**2
            aux_dist = np.sqrt(aux_dist)
            aux_ang = (gas_subhalo['Coordinates'][:,0]/1e3 - sub_pos_x[i]) * (halo_pos[0] - sub_pos_x[i]) + (gas_subhalo['Coordinates'][:,1]/1e3 - sub_pos_y[i]) * (halo_pos[1] - sub_pos_y[i]) + (gas_subhalo['Coordinates'][:,2]/1e3 - sub_pos_z[i]) * (halo_pos[2] - sub_pos_z[i])
            subhalo_ind_inside_2hmr = np.where(aux_dist < 2*sub_halfMassRad[i])[0]
            subhalo_ind_inside_fix = np.where(aux_dist < (50/1000))[0]
            subhalo_ind_bet_2_3 = np.where( (aux_dist > 2*sub_halfMassRad[i]) & (aux_dist < 3*sub_halfMassRad[i]) & (aux_ang > 0))[0] # entre 2 y 3 hmf, y en la cara cercana al cumulo

            subhalo_gas_num[i,0] = len(subhalo_ind_inside_fix)
            subhalo_gas_mass[i,0] = np.sum(gas_subhalo['Masses'][subhalo_ind_inside_fix])
            subhalo_gas_intEnergy[i,0] = np.sum(gas_subhalo['InternalEnergy'][subhalo_ind_inside_fix])
            subhalo_gas_elecAbundance[i,0] = np.sum(gas_subhalo['ElectronAbundance'][subhalo_ind_inside_fix])
            mu = (4 * gas_subhalo['Masses'][subhalo_ind_inside_fix]) / (1 + 3*0.76 + 4*0.76*gas_subhalo['ElectronAbundance'][subhalo_ind_inside_fix]) # auxiliar mu estimation
            subhalo_gas_T[i,0] = np.sum((5/3 - 1) * gas_subhalo['InternalEnergy'][subhalo_ind_inside_fix] * mu / K_B)

            subhalo_gas_num[i,1] = len(subhalo_ind_inside_2hmr)
            subhalo_gas_mass[i,1] = np.sum(gas_subhalo['Masses'][subhalo_ind_inside_2hmr])
            subhalo_gas_intEnergy[i,1] = np.sum(gas_subhalo['InternalEnergy'][subhalo_ind_inside_2hmr])
            subhalo_gas_elecAbundance[i,1] = np.sum(gas_subhalo['ElectronAbundance'][subhalo_ind_inside_2hmr])
            mu = (4 * gas_subhalo['Masses'][subhalo_ind_inside_2hmr]) / (1 + 3*0.76 + 4*0.76*gas_subhalo['ElectronAbundance'][subhalo_ind_inside_2hmr]) # auxiliar mu estimation
            subhalo_gas_T[i,1] = np.sum((5/3 - 1) * gas_subhalo['InternalEnergy'][subhalo_ind_inside_2hmr] * mu / K_B)

            subhalo_gas_num[i,2] = len(subhalo_ind_bet_2_3)
            subhalo_gas_mass[i,2] = np.sum(gas_subhalo['Masses'][subhalo_ind_bet_2_3])
            subhalo_gas_intEnergy[i,2] = np.sum(gas_subhalo['InternalEnergy'][subhalo_ind_bet_2_3])
            subhalo_gas_elecAbundance[i,2] = np.sum(gas_subhalo['ElectronAbundance'][subhalo_ind_bet_2_3])
            mu = (4 * gas_subhalo['Masses'][subhalo_ind_bet_2_3]) / (1 + 3*0.76 + 4*0.76*gas_subhalo['ElectronAbundance'][subhalo_ind_bet_2_3]) # auxiliar mu estimation
            subhalo_gas_T[i,2] = np.sum((5/3 - 1) * gas_subhalo['InternalEnergy'][subhalo_ind_bet_2_3] * mu / K_B)

            #print('El subhalo SI tiene gas en el snapshot ' + str(snap) + ' ' + str(len(subhalo_ind_inside_fix)))
            if plotear:
                rand_ind = np.random.randint(0, len(gas_subhalo[0]), 10000)
                ax.scatter(gas_subhalo[0][rand_ind,0]/1e3, gas_subhalo[0][rand_ind,1]/1e3, color = 'black', s = 5)
                ax.scatter(gas_subhalo[0][subhalo_ind_inside_fix,0]/1e3, gas_subhalo[0][subhalo_ind_inside_fix,1]/1e3, color = 'coral', s = 2, marker = '*')
                r200_circ = plt.Circle((sub_pos_x[i], sub_pos_y[i]), 2*sub_halfMassRad[i], color='coral', fill=False)
                ax.add_patch(r200_circ)
                
                path = 'Subhalo_' + str(sub_id_z0)
                os.makedirs(path, exist_ok=True)
                plt.savefig(path + '/Subhalo_' + str(sub_id_z0) + '_snap_' + str(snap) + '.pdf')
            #%gas_num_density[i] = (len(halo_ind_inside_2hmr) - len(subhalo_ind_inside_2hmr)) / len(subhalo_ind_inside_2hmr)
            #%gas_mass_density[i] = (gas_halo['masses'][halo_ind_inside_2hmr] - gas_subhalo['masses'][subhalo_ind_inside_2hmr]) / gas_subhalo['masses'][subhalo_ind_inside_2hmr]
        else:
            #print('El subhalo no tiene gas en el snapshot ' + str(snap))
            subhalo_gas_num[i,0] = 0
            subhalo_gas_mass[i,0] = 0
            subhalo_gas_intEnergy[i,0] = 0
            subhalo_gas_elecAbundance[i,0] = 0
            subhalo_gas_T[i,0] = 0
            
            subhalo_gas_num[i,1] = 0
            subhalo_gas_mass[i,1] = 0
            subhalo_gas_intEnergy[i,1] = 0
            subhalo_gas_elecAbundance[i,1] = 0
            subhalo_gas_T[i,1] = 0
            
            subhalo_gas_num[i,2] = 0
            subhalo_gas_mass[i,2] = 0
            subhalo_gas_intEnergy[i,2] = 0
            subhalo_gas_elecAbundance[i,2] = 0
            subhalo_gas_T[i,2] = 0

            if plotear:
                r200_circ = plt.Circle((sub_pos_x[i], sub_pos_y[i]), 2*sub_halfMassRad[i], color='coral', fill=False)
                ax.add_patch(r200_circ)
                
                path = 'Subhalo_' + str(sub_id_z0)
                os.makedirs(path, exist_ok=True)
                plt.savefig(path + '/Subhalo_' + str(sub_id_z0) + '_snap_' + str(snap) + '.pdf')
    return {
        'pos_x': np.array(pos_x_corrected),
        'pos_y': np.array(pos_y_corrected),
        'pos_z': np.array(pos_z_corrected),
        'snapshots': sub_snapshots,
        'halfMassRad': sub_halfMassRad,
        'Subhalo_gas_mass': subhalo_gas_mass,
        'Subhalo_gas_num': subhalo_gas_num,
        'Subhalo_gas_T': subhalo_gas_T,
        'Halo_gas_mass': halo_gas_mass,
        'Halo_gas_num': halo_gas_num,
        'Halo_gas_T': halo_gas_T
    }

    #except Exception as e:
    #    return None

In [20]:
def get(path, params=None, folderName=''):
    '''
    Illustris function
    '''
    # make HTTP GET request to path
    r = requests.get(path, params=params, headers=headers)

    # raise exception if response code is not HTTP SUCCESS (200)
    r.raise_for_status()

    if r.headers['content-type'] == 'application/json':
        return r.json() # parse json responses automatically

    if 'content-disposition' in r.headers:
        filename = r.headers['content-disposition'].split("filename=")[1]
        if filename.endswith('.hdf5'):
            file_access_property_list = h5py.h5p.create(h5py.h5p.FILE_ACCESS)
            file_access_property_list.set_fapl_core(backing_store=False)
            file_access_property_list.set_file_image(r.content)
            
            file_id_args = {
                'fapl': file_access_property_list,
                'flags': h5py.h5f.ACC_RDONLY,
                'name': next(tempfile._get_candidate_names()).encode()
            }
            
            h5_file_args = {'backing_store': False, 'driver': 'core', 'mode': 'r'}
            with contextlib.closing(h5py.h5f.open(**file_id_args)) as file_id:
                with h5py.File(file_id, **h5_file_args) as h5_file:
                    #return np.array(h5_file['grid'])
                    if 'grid' in h5_file.keys(): return np.array(h5_file['grid'])
                    else:
                        results = []
                        for k in h5_file.keys():
                            for sk in h5_file[k].keys():
                                results.append(np.array(h5_file[k][sk]))
                        return results
        else:
            with open(folderName + filename, 'wb') as f:
                f.write(r.content)
            return filename # return the filename string
    return r

def get_url(which, subhalo_id, snap):
    return 'http://www.tng-project.org/api/TNG300-1/snapshots/' + str(snap) + '/' + which + '/' + str(subhalo_id)

headers = {"api-key": '81b7c70637fa8b110e6b9f236ea07c37'}

In [21]:
%%time
# Crear el archivo HDF5
with h5py.File('data_tree_halo3.hdf5', 'w') as hdf5_file:
    
    # Agrupar por halo_id_z0 para procesar cada halo una sola vez
    grouped_data = data.groupby('halo_id_z0')
    
    # Iterar sobre cada halo
    contador_halos = 0
    for halo_id_z0, halo_subhalos in tqdm(grouped_data, desc='Halos', total=len(grouped_data)):
        halo_id_z0 = int(halo_id_z0)
        print('Analizando Halo ' + str(halo_id_z0))
        if (contador_halos < 1) & (halo_id_z0 > 2):
            contador_halos += 1
            # Obtener el ID de la galaxia central (asumimos que es el mismo para todos los subhalos del halo)
            id_central_galaxy_z0 = int(halo_subhalos.iloc[0]['halo_id_central_galaxy_z0'])

            try:
                # Cargar evolución del halo
                halo_evolution = load_halo_evolution(myBasePath, halo_id_z0, id_central_galaxy_z0)

                if halo_evolution is None:
                    continue

                # Crear grupo para este halo
                halo_group = hdf5_file.create_group(f'halo_{halo_id_z0}')

                # Guardar datos del halo
                halo_group.create_dataset('pos_x', data=halo_evolution['pos_x'])
                halo_group.create_dataset('pos_y', data=halo_evolution['pos_y'])
                halo_group.create_dataset('pos_z', data=halo_evolution['pos_z'])
                halo_group.create_dataset('r200', data=halo_evolution['r200'])
                halo_group.create_dataset('m200', data=halo_evolution['m200'])
                halo_group.create_dataset('snapshots', data=halo_evolution['snapshots'])

                # Procesar cada subhalo de este halo
                contador_subhalos = 0
                for _, row in halo_subhalos[:].iterrows():
                    if (contador_subhalos < 1000):
                        contador_subhalos += 1
                        sub_id_z0 = int(row['sub_id_z0'])

                        try:
                            # Cargar evolución del subhalo con correcciones de periodicidad
                            subhalo_evolution = load_subhalo_evolution(myBasePath, sub_id_z0, halo_evolution, plotear = False)

                            if subhalo_evolution is None:
                                continue

                            # Crear subgrupo para este subhalo dentro del halo
                            subhalo_group = halo_group.create_group(f'subhalo_{sub_id_z0}')

                            # Guardar datos del subhalo (ya corregidos por periodicidad)
                            subhalo_group.create_dataset('pos_x', data=subhalo_evolution['pos_x'])
                            subhalo_group.create_dataset('pos_y', data=subhalo_evolution['pos_y'])
                            subhalo_group.create_dataset('pos_z', data=subhalo_evolution['pos_z'])
                            subhalo_group.create_dataset('snapshots', data=subhalo_evolution['snapshots'])
                            subhalo_group.create_dataset('halfMassRad', data = subhalo_evolution['halfMassRad'])
                            subhalo_group.create_dataset('Subhalo_gas_mass', data = subhalo_evolution['Subhalo_gas_mass'])
                            subhalo_group.create_dataset('Subhalo_gas_num', data = subhalo_evolution['Subhalo_gas_num'])
                            subhalo_group.create_dataset('Subhalo_gas_T', data = subhalo_evolution['Subhalo_gas_T'])
                            subhalo_group.create_dataset('Halo_gas_mass', data = subhalo_evolution['Halo_gas_mass'])
                            subhalo_group.create_dataset('Halo_gas_num', data = subhalo_evolution['Halo_gas_num'])
                            subhalo_group.create_dataset('Halo_gas_T', data = subhalo_evolution['Halo_gas_T'])

                        except Exception as e:
                            continue

            except Exception as e:
                continue

Halos:   0%|          | 0/549 [00:00<?, ?it/s]

Analizando Halo 0
Analizando Halo 2
Analizando Halo 4
Snapshot 99. Subhalo 27878
Snapshot 89. Subhalo 27878
Snapshot 79. Subhalo 27878
Snapshot 69. Subhalo 27878
Snapshot 59. Subhalo 27878
Snapshot 49. Subhalo 27878
Snapshot 39. Subhalo 27878
Snapshot 29. Subhalo 27878
Snapshot 19. Subhalo 27878
Snapshot 9. Subhalo 27878
Snapshot 99. Subhalo 27879
Snapshot 89. Subhalo 27879
Snapshot 79. Subhalo 27879
Snapshot 69. Subhalo 27879
Snapshot 59. Subhalo 27879
Snapshot 49. Subhalo 27879
Snapshot 39. Subhalo 27879
Snapshot 29. Subhalo 27879
Snapshot 19. Subhalo 27879
Snapshot 9. Subhalo 27879
Snapshot 99. Subhalo 27880
Snapshot 89. Subhalo 27880
Snapshot 79. Subhalo 27880
Snapshot 69. Subhalo 27880
Snapshot 59. Subhalo 27880
Snapshot 49. Subhalo 27880
Snapshot 39. Subhalo 27880
Snapshot 29. Subhalo 27880
Snapshot 19. Subhalo 27880
Snapshot 9. Subhalo 27880
Snapshot 99. Subhalo 27881
Snapshot 89. Subhalo 27881
Snapshot 79. Subhalo 27881
Snapshot 69. Subhalo 27881
Snapshot 59. Subhalo 27881
Snap

Halos:   1%|          | 3/549 [38:01:51<6921:37:45, 45637.12s/it]

Analizando Halo 6
Analizando Halo 7
Analizando Halo 8
Analizando Halo 9
Analizando Halo 10
Analizando Halo 12
Analizando Halo 14
Analizando Halo 15
Analizando Halo 16
Analizando Halo 17
Analizando Halo 19
Analizando Halo 20
Analizando Halo 21
Analizando Halo 22
Analizando Halo 23
Analizando Halo 24
Analizando Halo 25
Analizando Halo 26
Analizando Halo 28
Analizando Halo 29
Analizando Halo 30
Analizando Halo 32
Analizando Halo 33
Analizando Halo 37
Analizando Halo 38
Analizando Halo 39
Analizando Halo 40
Analizando Halo 41
Analizando Halo 42
Analizando Halo 43
Analizando Halo 45
Analizando Halo 46
Analizando Halo 47
Analizando Halo 48
Analizando Halo 49
Analizando Halo 50
Analizando Halo 53
Analizando Halo 54
Analizando Halo 55
Analizando Halo 56
Analizando Halo 57
Analizando Halo 59
Analizando Halo 60
Analizando Halo 61
Analizando Halo 62
Analizando Halo 65
Analizando Halo 66
Analizando Halo 67
Analizando Halo 68
Analizando Halo 70
Analizando Halo 71
Analizando Halo 72
Analizando Halo 

Halos: 100%|██████████| 549/549 [38:01:54<00:00, 249.39s/it]     

Analizando Halo 288
Analizando Halo 290
Analizando Halo 291
Analizando Halo 295
Analizando Halo 296
Analizando Halo 298
Analizando Halo 299
Analizando Halo 300
Analizando Halo 301
Analizando Halo 302
Analizando Halo 305
Analizando Halo 306
Analizando Halo 307
Analizando Halo 308
Analizando Halo 309
Analizando Halo 310
Analizando Halo 311
Analizando Halo 312
Analizando Halo 314
Analizando Halo 317
Analizando Halo 318
Analizando Halo 319
Analizando Halo 321
Analizando Halo 322
Analizando Halo 326
Analizando Halo 327
Analizando Halo 329
Analizando Halo 331
Analizando Halo 332
Analizando Halo 333
Analizando Halo 335
Analizando Halo 336
Analizando Halo 337
Analizando Halo 338
Analizando Halo 340
Analizando Halo 341
Analizando Halo 342
Analizando Halo 343
Analizando Halo 344
Analizando Halo 345
Analizando Halo 348
Analizando Halo 349
Analizando Halo 350
Analizando Halo 351
Analizando Halo 353
Analizando Halo 354
Analizando Halo 355
Analizando Halo 356
Analizando Halo 357
Analizando Halo 358


CPU times: user 3h 14min 53s, sys: 18h 55min 8s, total: 22h 10min 2s
Wall time: 1d 14h 1min 55s


In [22]:
## 22hs para halo 3

1

In [ ]:
%%time
# Crear el archivo HDF5
with h5py.File('data_tree_halo3.hdf5', 'w') as hdf5_file:
    
    # Agrupar por halo_id_z0 para procesar cada halo una sola vez
    grouped_data = data.groupby('halo_id_z0')
    
    # Iterar sobre cada halo
    contador_halos = 0
    for halo_id_z0, halo_subhalos in tqdm(grouped_data, desc='Halos', total=len(grouped_data)):
        halo_id_z0 = int(halo_id_z0)
        print('Analizando Halo ' + str(halo_id_z0))
        if (contador_halos < 1) & (halo_id_z0 > 2):
            contador_halos += 1
            # Obtener el ID de la galaxia central (asumimos que es el mismo para todos los subhalos del halo)
            id_central_galaxy_z0 = int(halo_subhalos.iloc[0]['halo_id_central_galaxy_z0'])

            #try:
            # Cargar evolución del halo
            halo_evolution = load_halo_evolution(myBasePath, halo_id_z0, id_central_galaxy_z0)

            if halo_evolution is None:
                continue

            # Crear grupo para este halo
            halo_group = hdf5_file.create_group(f'halo_{halo_id_z0}')

            # Guardar datos del halo
            halo_group.create_dataset('pos_x', data=halo_evolution['pos_x'])
            halo_group.create_dataset('pos_y', data=halo_evolution['pos_y'])
            halo_group.create_dataset('pos_z', data=halo_evolution['pos_z'])
            halo_group.create_dataset('r200', data=halo_evolution['r200'])
            halo_group.create_dataset('m200', data=halo_evolution['m200'])
            halo_group.create_dataset('snapshots', data=halo_evolution['snapshots'])

            # Procesar cada subhalo de este halo
            contador = 0
            for i, row in halo_subhalos[:].iterrows():
                sub_id_z0 = int(row['sub_id_z0'])
                tree = il.sublink.loadTree(
                    myBasePath, 99, sub_id_z0, 
                    fields=['SubhaloGrNr', 'SubhaloPos', 'SnapNum','SubhaloHalfmassRad','SubhaloIDRaw'], 
                    onlyMPB=True
                )
                if tree is not None:
                    aux_array = np.zeros((tree['count'], 8))

                    aux_array[:,0] = sub_id_z0
                    aux_array[:,1] = tree['SubhaloGrNr']
                    aux_array[:,2] = tree['SubhaloPos'][:,0]
                    aux_array[:,3] = tree['SubhaloPos'][:,1]
                    aux_array[:,4] = tree['SubhaloPos'][:,2]
                    aux_array[:,5] = tree['SnapNum']
                    aux_array[:,6] = tree['SubhaloHalfmassRad']
                    aux_array[:,7] = tree['SubhaloIDRaw']

                    if contador == 0:
                        subhalos_list = np.copy(aux_array)
                    if contador > 0:
                        subhalos_list = np.vstack((subhalos_list, aux_array))
                    contador += 1
                    
            print(subhalos_list.shape)
            # Arrays para posiciones corregidas
            pos_x_corrected = []
            pos_y_corrected = []
            pos_z_corrected = []

            subhalo_gas_num = np.ones((len(subhalos_list), 3)) * (-99) 
            halo_gas_num = np.ones((len(subhalos_list), 3)) * (-99)
            subhalo_gas_mass = np.ones((len(subhalos_list), 3)) * (-99) 
            halo_gas_mass = np.ones((len(subhalos_list), 3)) * (-99) 
            subhalo_gas_intEnergy = np.ones((len(subhalos_list), 3)) * (-99) 
            halo_gas_intEnergy = np.ones((len(subhalos_list), 3)) * (-99) 
            subhalo_gas_elecAbundance = np.ones((len(subhalos_list), 3)) * (-99) 
            halo_gas_elecAbundance = np.ones((len(subhalos_list), 3)) * (-99)
            subhalo_gas_T = np.ones((len(subhalos_list), 3)) * (-99) 
            halo_gas_T = np.ones((len(subhalos_list), 3)) * (-99)
            
            subhalo_pos = np.ones((len(subhalos_list), 3)) * (-99)
            subhalo_hmr = np.ones(len(subhalos_list)) * (-99)

            for isnap, snap in enumerate(halo_evolution['snapshots']):
                print(snap)
                # Encontrar el índice correspondiente en halo_evolution
                halo_idx = np.where(halo_evolution['snapshots'] == snap)[0]
                haloID = halo_evolution['haloID'][halo_idx[0]]

                halo_idx = halo_idx[0]

                # Posición del halo en este snapshot
                halo_pos = np.array([
                    halo_evolution['pos_x'][halo_idx],
                    halo_evolution['pos_y'][halo_idx],
                    halo_evolution['pos_z'][halo_idx]
                ])

                # Me guardo todas las particulas de gas del halo
                gas_halo = il.snapshot.loadHalo(myBasePath, snap, haloID, 'gas',
                                                      fields = ['Coordinates','Density','ElectronAbundance','InternalEnergy','Masses'])

                ind_snap = np.where(subhalos_list[:,5] == snap)[0]
                for i in ind_snap[:]:
                    # Posición del subhalo en este snapshot
                    sub_pos = np.array([subhalos_list[i, 2]/1e3, subhalos_list[i, 3]/1e3, subhalos_list[i, 4]/1e3])
                    subhalo_pos[i,:] = sub_pos
                    # Aplicar corrección de periodicidad
                    corrected_pos = apply_periodic_correction(sub_pos, halo_pos, box_size)

                    pos_x_corrected.append(corrected_pos[0])
                    pos_y_corrected.append(corrected_pos[1])
                    pos_z_corrected.append(corrected_pos[2])


                    # Me guardo todas las particulas de gas del subhalo (el subhaloIDRaw es el subfindID + snapnum*1e12)
                    gas_subhalo = il.snapshot.loadSubhalo(myBasePath, snap, int(subhalos_list[i,7] - snap*1e12), 'gas',
                                                          fields = ['Coordinates','Density','ElectronAbundance','InternalEnergy','Masses'])


                    if (gas_halo['count'] > 0): # Si no hay gas en el subhalo no hago nada
                        # Me quedo con todo lo que este adentro de 2* galaxy-half-mass-radii
                        aux_dist = (gas_halo['Coordinates'][:,0]/1e3 - sub_pos[0])**2 + (gas_halo['Coordinates'][:,1]/1e3 - sub_pos[1])**2 + (gas_halo['Coordinates'][:,2]/1e3 - sub_pos[2])**2
                        aux_dist = np.sqrt(aux_dist) # MPc/h
                        aux_ang = (gas_halo['Coordinates'][:,0]/1e3 - sub_pos[0]) * (halo_pos[0] - sub_pos[0]) + (gas_halo['Coordinates'][:,1]/1e3 - sub_pos[1]) * (halo_pos[1] - sub_pos[1]) + (gas_halo['Coordinates'][:,2]/1e3 - sub_pos[2]) * (halo_pos[2] - sub_pos[2])
                        halo_ind_inside_2hmr = np.where(aux_dist < 2*subhalos_list[i,6])[0]
                        halo_ind_inside_fix = np.where(aux_dist < (50/1000))[0]
                        halo_ind_bet_2_3 = np.where( (aux_dist > 2*subhalos_list[i,6]) & (aux_dist < 3*subhalos_list[i,6]) & (aux_ang > 0) )[0] # entre 2 y 3 hmf, y en la cara cercana al cumulo

                        halo_gas_num[i,0] = len(halo_ind_inside_fix)
                        halo_gas_mass[i,0] = np.sum(gas_halo['Masses'][halo_ind_inside_fix])
                        halo_gas_intEnergy[i,0] = np.sum(gas_halo['InternalEnergy'][halo_ind_inside_fix])
                        halo_gas_elecAbundance[i,0] = np.sum(gas_halo['ElectronAbundance'][halo_ind_inside_fix])
                        mu = (4 * gas_halo['Masses'][halo_ind_inside_fix]) / (1 + 3*0.76 + 4*0.76*gas_halo['ElectronAbundance'][halo_ind_inside_fix]) # auxiliar mu estimation
                        halo_gas_T[i,0] = np.sum((5/3 - 1) * gas_halo['InternalEnergy'][halo_ind_inside_fix] * mu / K_B)

                        halo_gas_num[i,1] = len(halo_ind_inside_2hmr)
                        halo_gas_mass[i,1] = np.sum(gas_halo['Masses'][halo_ind_inside_2hmr])
                        halo_gas_intEnergy[i,1] = np.sum(gas_halo['InternalEnergy'][halo_ind_inside_2hmr])
                        halo_gas_elecAbundance[i,1] = np.sum(gas_halo['ElectronAbundance'][halo_ind_inside_2hmr])
                        mu = (4 * gas_halo['Masses'][halo_ind_inside_2hmr]) / (1 + 3*0.76 + 4*0.76*gas_halo['ElectronAbundance'][halo_ind_inside_2hmr]) # auxiliar mu estimation
                        halo_gas_T[i,1] = np.sum((5/3 - 1) * gas_halo['InternalEnergy'][halo_ind_inside_2hmr] * mu / K_B)

                        halo_gas_num[i,2] = len(halo_ind_bet_2_3)
                        halo_gas_mass[i,2] = np.sum(gas_halo['Masses'][halo_ind_bet_2_3])
                        halo_gas_intEnergy[i,2] = np.sum(gas_halo['InternalEnergy'][halo_ind_bet_2_3])
                        halo_gas_elecAbundance[i,2] = np.sum(gas_halo['ElectronAbundance'][halo_ind_bet_2_3])
                        mu = (4 * gas_halo['Masses'][halo_ind_bet_2_3]) / (1 + 3*0.76 + 4*0.76*gas_halo['ElectronAbundance'][halo_ind_bet_2_3]) # auxiliar mu estimation
                        halo_gas_T[i,2] = np.sum((5/3 - 1) * gas_halo['InternalEnergy'][halo_ind_bet_2_3] * mu / K_B)

                    else:
                        halo_gas_num[i,0] = 0
                        halo_gas_mass[i,0] = 0
                        halo_gas_intEnergy[i,0] = 0
                        halo_gas_elecAbundance[i,0] = 0
                        halo_gas_T[i,0] = 0

                        halo_gas_num[i,1] = 0
                        halo_gas_mass[i,1] = 0
                        halo_gas_intEnergy[i,1] = 0
                        halo_gas_elecAbundance[i,1] = 0
                        halo_gas_T[i,1] = 0

                        halo_gas_num[i,2] = 0
                        halo_gas_mass[i,2] = 0
                        halo_gas_intEnergy[i,2] = 0
                        halo_gas_elecAbundance[i,2] = 0
                        halo_gas_T[i,2] = 0

                    if (gas_subhalo['count'] > 0):
                        # Me quedo con todo lo que este adentro de 2* galaxy-half-mass-radii
                        aux_dist = (gas_subhalo['Coordinates'][:,0]/1e3 - sub_pos[0])**2 + (gas_subhalo['Coordinates'][:,1]/1e3 - sub_pos[1])**2 + (gas_subhalo['Coordinates'][:,2]/1e3 - sub_pos[2])**2
                        aux_dist = np.sqrt(aux_dist)
                        aux_ang = (gas_subhalo['Coordinates'][:,0]/1e3 - sub_pos[0]) * (halo_pos[0] - sub_pos[0]) + (gas_subhalo['Coordinates'][:,1]/1e3 - sub_pos[1]) * (halo_pos[1] - sub_pos[1]) + (gas_subhalo['Coordinates'][:,2]/1e3 - sub_pos[2]) * (halo_pos[2] - sub_pos[2])
                        subhalo_ind_inside_2hmr = np.where(aux_dist < 2*subhalos_list[i,6])[0]
                        subhalo_ind_inside_fix = np.where(aux_dist < (50/1000))[0]
                        subhalo_ind_bet_2_3 = np.where( (aux_dist > 2*subhalos_list[i,6]) & (aux_dist < 3*subhalos_list[i,6]) & (aux_ang > 0))[0] # entre 2 y 3 hmf, y en la cara cercana al cumulo

                        subhalo_gas_num[i,0] = len(subhalo_ind_inside_fix)
                        subhalo_gas_mass[i,0] = np.sum(gas_subhalo['Masses'][subhalo_ind_inside_fix])
                        subhalo_gas_intEnergy[i,0] = np.sum(gas_subhalo['InternalEnergy'][subhalo_ind_inside_fix])
                        subhalo_gas_elecAbundance[i,0] = np.sum(gas_subhalo['ElectronAbundance'][subhalo_ind_inside_fix])
                        mu = (4 * gas_subhalo['Masses'][subhalo_ind_inside_fix]) / (1 + 3*0.76 + 4*0.76*gas_subhalo['ElectronAbundance'][subhalo_ind_inside_fix]) # auxiliar mu estimation
                        subhalo_gas_T[i,0] = np.sum((5/3 - 1) * gas_subhalo['InternalEnergy'][subhalo_ind_inside_fix] * mu / K_B)

                        subhalo_gas_num[i,1] = len(subhalo_ind_inside_2hmr)
                        subhalo_gas_mass[i,1] = np.sum(gas_subhalo['Masses'][subhalo_ind_inside_2hmr])
                        subhalo_gas_intEnergy[i,1] = np.sum(gas_subhalo['InternalEnergy'][subhalo_ind_inside_2hmr])
                        subhalo_gas_elecAbundance[i,1] = np.sum(gas_subhalo['ElectronAbundance'][subhalo_ind_inside_2hmr])
                        mu = (4 * gas_subhalo['Masses'][subhalo_ind_inside_2hmr]) / (1 + 3*0.76 + 4*0.76*gas_subhalo['ElectronAbundance'][subhalo_ind_inside_2hmr]) # auxiliar mu estimation
                        subhalo_gas_T[i,1] = np.sum((5/3 - 1) * gas_subhalo['InternalEnergy'][subhalo_ind_inside_2hmr] * mu / K_B)

                        subhalo_gas_num[i,2] = len(subhalo_ind_bet_2_3)
                        subhalo_gas_mass[i,2] = np.sum(gas_subhalo['Masses'][subhalo_ind_bet_2_3])
                        subhalo_gas_intEnergy[i,2] = np.sum(gas_subhalo['InternalEnergy'][subhalo_ind_bet_2_3])
                        subhalo_gas_elecAbundance[i,2] = np.sum(gas_subhalo['ElectronAbundance'][subhalo_ind_bet_2_3])
                        mu = (4 * gas_subhalo['Masses'][subhalo_ind_bet_2_3]) / (1 + 3*0.76 + 4*0.76*gas_subhalo['ElectronAbundance'][subhalo_ind_bet_2_3]) # auxiliar mu estimation
                        subhalo_gas_T[i,2] = np.sum((5/3 - 1) * gas_subhalo['InternalEnergy'][subhalo_ind_bet_2_3] * mu / K_B)

                    else:
                        #print('El subhalo no tiene gas en el snapshot ' + str(snap))
                        subhalo_gas_num[i,0] = 0
                        subhalo_gas_mass[i,0] = 0
                        subhalo_gas_intEnergy[i,0] = 0
                        subhalo_gas_elecAbundance[i,0] = 0
                        subhalo_gas_T[i,0] = 0

                        subhalo_gas_num[i,1] = 0
                        subhalo_gas_mass[i,1] = 0
                        subhalo_gas_intEnergy[i,1] = 0
                        subhalo_gas_elecAbundance[i,1] = 0
                        subhalo_gas_T[i,1] = 0

                        subhalo_gas_num[i,2] = 0
                        subhalo_gas_mass[i,2] = 0
                        subhalo_gas_intEnergy[i,2] = 0
                        subhalo_gas_elecAbundance[i,2] = 0
                        subhalo_gas_T[i,2] = 0


                
                
    print('guardando')            
    for sub_ind in np.unique(subhalos_list[:,0])[:10]:          
        aux_ind = np.where(subhalos_list[:,0] == sub_ind)[0]
        # Crear subgrupo para este subhalo dentro del halo
        subhalo_group = halo_group.create_group(f'subhalo_{int(sub_ind)}')

        # Guardar datos del subhalo (ya corregidos por periodicidad)
        subhalo_group.create_dataset('Subhalo_pos', data = subhalo_pos[aux_ind])
        subhalo_group.create_dataset('pos_x', data = subhalos_list[aux_ind,2]/1e3)
        subhalo_group.create_dataset('pos_y', data = subhalos_list[aux_ind,3]/1e3)
        subhalo_group.create_dataset('pos_z', data = subhalos_list[aux_ind,4]/1e3)
        subhalo_group.create_dataset('halfMassRad', data = subhalos_list[aux_ind,6])
        subhalo_group.create_dataset('Subhalo_gas_mass', data = subhalo_gas_mass[aux_ind])
        subhalo_group.create_dataset('Subhalo_gas_num', data = subhalo_gas_num[aux_ind])
        subhalo_group.create_dataset('Subhalo_gas_T', data = subhalo_gas_T[aux_ind])
        subhalo_group.create_dataset('Halo_gas_mass', data = halo_gas_mass[aux_ind])
        subhalo_group.create_dataset('Halo_gas_num', data = halo_gas_num[aux_ind])
        subhalo_group.create_dataset('Halo_gas_T', data = halo_gas_T[aux_ind])

Halos:   0%|          | 0/549 [00:00<?, ?it/s]

Analizando Halo 0
Analizando Halo 2
Analizando Halo 4
Warning, empty return. Subhalo [28282] at snapNum [99] not in tree.
(51513, 8)
99
98
97
96
95
94
93
92
91
90
89
88
87
86


In [122]:
%%time
# Crear el archivo HDF5
with h5py.File('data_tree_halo3_2.hdf5', 'w') as hdf5_file:
    
    # Agrupar por halo_id_z0 para procesar cada halo una sola vez
    grouped_data = data.groupby('halo_id_z0')
    
    # Iterar sobre cada halo
    contador_halos = 0
    for halo_id_z0, halo_subhalos in tqdm(grouped_data, desc='Halos', total=len(grouped_data)):
        halo_id_z0 = int(halo_id_z0)
        print('Analizando Halo ' + str(halo_id_z0))
        if (contador_halos < 1) & (halo_id_z0 > 2):
            contador_halos += 1
            # Obtener el ID de la galaxia central (asumimos que es el mismo para todos los subhalos del halo)
            id_central_galaxy_z0 = int(halo_subhalos.iloc[0]['halo_id_central_galaxy_z0'])

            try:
                # Cargar evolución del halo
                halo_evolution = load_halo_evolution(myBasePath, halo_id_z0, id_central_galaxy_z0)

                if halo_evolution is None:
                    continue

                # Crear grupo para este halo
                halo_group = hdf5_file.create_group(f'halo_{halo_id_z0}')

                # Guardar datos del halo
                halo_group.create_dataset('pos_x', data=halo_evolution['pos_x'])
                halo_group.create_dataset('pos_y', data=halo_evolution['pos_y'])
                halo_group.create_dataset('pos_z', data=halo_evolution['pos_z'])
                halo_group.create_dataset('r200', data=halo_evolution['r200'])
                halo_group.create_dataset('m200', data=halo_evolution['m200'])
                halo_group.create_dataset('snapshots', data=halo_evolution['snapshots'])

                # Procesar cada subhalo de este halo
                contador_subhalos = 0
                for _, row in halo_subhalos[:].iterrows():
                    if (contador_subhalos < 10):
                        contador_subhalos += 1
                        sub_id_z0 = int(row['sub_id_z0'])

                        try:
                            # Cargar evolución del subhalo con correcciones de periodicidad
                            subhalo_evolution = load_subhalo_evolution(myBasePath, sub_id_z0, halo_evolution, plotear = False)

                            if subhalo_evolution is None:
                                continue

                            # Crear subgrupo para este subhalo dentro del halo
                            subhalo_group = halo_group.create_group(f'subhalo_{sub_id_z0}')

                            # Guardar datos del subhalo (ya corregidos por periodicidad)
                            subhalo_group.create_dataset('pos_x', data=subhalo_evolution['pos_x'])
                            subhalo_group.create_dataset('pos_y', data=subhalo_evolution['pos_y'])
                            subhalo_group.create_dataset('pos_z', data=subhalo_evolution['pos_z'])
                            subhalo_group.create_dataset('snapshots', data=subhalo_evolution['snapshots'])
                            subhalo_group.create_dataset('halfMassRad', data = subhalo_evolution['halfMassRad'])
                            subhalo_group.create_dataset('Subhalo_gas_mass', data = subhalo_evolution['Subhalo_gas_mass'])
                            subhalo_group.create_dataset('Subhalo_gas_num', data = subhalo_evolution['Subhalo_gas_num'])
                            subhalo_group.create_dataset('Subhalo_gas_T', data = subhalo_evolution['Subhalo_gas_T'])
                            subhalo_group.create_dataset('Halo_gas_mass', data = subhalo_evolution['Halo_gas_mass'])
                            subhalo_group.create_dataset('Halo_gas_num', data = subhalo_evolution['Halo_gas_num'])
                            subhalo_group.create_dataset('Halo_gas_T', data = subhalo_evolution['Halo_gas_T'])

                        except Exception as e:
                            continue

            except Exception as e:
                continue

Halos:   0%|          | 1/549 [00:00<01:49,  5.03it/s]

Analizando Halo 0
Analizando Halo 2
Analizando Halo 4
Snapshot 99. Subhalo 27878
Snapshot 89. Subhalo 27878
Snapshot 79. Subhalo 27878
Snapshot 69. Subhalo 27878
Snapshot 59. Subhalo 27878
Snapshot 49. Subhalo 27878
Snapshot 39. Subhalo 27878
Snapshot 29. Subhalo 27878
Snapshot 19. Subhalo 27878
Snapshot 9. Subhalo 27878
Snapshot 99. Subhalo 27879
Snapshot 89. Subhalo 27879
Snapshot 79. Subhalo 27879
Snapshot 69. Subhalo 27879
Snapshot 59. Subhalo 27879
Snapshot 49. Subhalo 27879
Snapshot 39. Subhalo 27879
Snapshot 29. Subhalo 27879
Snapshot 19. Subhalo 27879
Snapshot 9. Subhalo 27879
Snapshot 99. Subhalo 27880
Snapshot 89. Subhalo 27880
Snapshot 79. Subhalo 27880
Snapshot 69. Subhalo 27880
Snapshot 59. Subhalo 27880
Snapshot 49. Subhalo 27880
Snapshot 39. Subhalo 27880
Snapshot 29. Subhalo 27880
Snapshot 19. Subhalo 27880
Snapshot 9. Subhalo 27880
Snapshot 99. Subhalo 27881
Snapshot 89. Subhalo 27881
Snapshot 79. Subhalo 27881
Snapshot 69. Subhalo 27881
Snapshot 59. Subhalo 27881
Snap

Halos: 100%|██████████| 549/549 [58:58<00:00,  6.45s/it]    

Analizando Halo 6
Analizando Halo 7
Analizando Halo 8
Analizando Halo 9
Analizando Halo 10
Analizando Halo 12
Analizando Halo 14
Analizando Halo 15
Analizando Halo 16
Analizando Halo 17
Analizando Halo 19
Analizando Halo 20
Analizando Halo 21
Analizando Halo 22
Analizando Halo 23
Analizando Halo 24
Analizando Halo 25
Analizando Halo 26
Analizando Halo 28
Analizando Halo 29
Analizando Halo 30
Analizando Halo 32
Analizando Halo 33
Analizando Halo 37
Analizando Halo 38
Analizando Halo 39
Analizando Halo 40
Analizando Halo 41
Analizando Halo 42
Analizando Halo 43
Analizando Halo 45
Analizando Halo 46
Analizando Halo 47
Analizando Halo 48
Analizando Halo 49
Analizando Halo 50
Analizando Halo 53
Analizando Halo 54
Analizando Halo 55
Analizando Halo 56
Analizando Halo 57
Analizando Halo 59
Analizando Halo 60
Analizando Halo 61
Analizando Halo 62
Analizando Halo 65
Analizando Halo 66
Analizando Halo 67
Analizando Halo 68
Analizando Halo 70
Analizando Halo 71
Analizando Halo 72
Analizando Halo 

# PASO 5: ELIMINAR SUBHALOS SIN MERGER TREE

In [337]:
valid_sub_ids = []

with h5py.File('data_tree_halo3_3.hdf5', "r") as f:
    for halo_name in f.keys():
        halo_group = f[halo_name]

        # Buscar subhalos
        subhalos = [k for k in halo_group.keys() if k.startswith("subhalo_")]

        # Extraer el número de cada subhalo y guardarlo
        for sub_name in subhalos:
            sub_id = int(sub_name.split("_")[1])
            valid_sub_ids.append(sub_id)

# Convertimos a set para buscar rápido
valid_sub_ids = set(valid_sub_ids)

print(f"Total subhalos en merger tree: {len(valid_sub_ids)}")
print(f"Primeros 20 ids: {list(valid_sub_ids)[:20]}")

Total subhalos en merger tree: 543
Primeros 20 ids: [1597448, 1597449, 1933338, 2291753, 2293817, 2201658, 2138197, 2138198, 2363489, 1104033, 1104034, 1104035, 1104036, 1104037, 1867940, 1867941, 1396904, 1396905, 1396906, 1396907]


In [338]:
data_clean = data[data["sub_id_z0"].isin(valid_sub_ids)].copy()

print(f"Subhalos originales en DataFrame: {len(data)}")
print(f"Subhalos después de limpieza: {len(data_clean)}")

Subhalos originales en DataFrame: 39132
Subhalos después de limpieza: 665


In [339]:
# Eliminamos duplicados
data_clean_final = data_clean.drop_duplicates(subset=['sub_id_z0'])
print(len(data_clean_final))

543


In [340]:
data_clean_final.to_csv('link_halo_subhalos_z0_halo3_3.dat', sep=' ', index=False)